# Bloque 2: LLMs y RAG — Prompt Engineering en Contexto Inmobiliario

En este bloque aprenderemos dos habilidades clave para cualquier equipo que quiera aprovechar la IA generativa en su producto:

1. **Prompt Engineering**: cómo hablarle al modelo para obtener respuestas útiles, predecibles y estructuradas.
2. **RAG (Retrieval Augmented Generation)**: cómo dar al modelo acceso a *nuestros propios datos* en tiempo real, sin necesidad de reentrenar nada.

Usaremos ejemplos concretos del mundo inmobiliario para que cada patrón tenga sentido de negocio real.

> **Nota técnica**: este notebook llama a la API de Anthropic (Claude). Necesitas tener la variable de entorno `ANTHROPIC_API_KEY` configurada antes de ejecutarlo.

## Configuración inicial

Importamos las librerías necesarias y creamos el cliente de la API. El cliente es el objeto que nos permite enviar mensajes a Claude y recibir sus respuestas.

In [1]:
import os
import json
import math
import re
from collections import Counter
from dotenv import load_dotenv
import anthropic

# Cargar .env — busca en el directorio del notebook y en la carpeta padre (raíz del proyecto)
load_dotenv()
load_dotenv(dotenv_path=os.path.join(os.path.dirname(os.getcwd()), ".env"))

MODELO = "claude-sonnet-4-6"

api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise EnvironmentError(
        "No se encontró ANTHROPIC_API_KEY.\n"
        "Opciones:\n"
        "  1. Crea un archivo .env en esta carpeta (notebooks/.env) con: ANTHROPIC_API_KEY=sk-ant-...\n"
        "  2. O en la terminal antes de abrir Jupyter: set ANTHROPIC_API_KEY=sk-ant-..."
    )

cliente = anthropic.Anthropic(api_key=api_key)
print(f"✓ Cliente configurado. Modelo: {MODELO}")

✓ Cliente configurado. Modelo: claude-sonnet-4-6


## Parte 1: Prompt Engineering — 5 Patrones Fundamentales

Un **prompt** es el texto que le enviamos al modelo. La calidad del prompt determina directamente la calidad de la respuesta.

Estos 5 patrones son los que más impacto tienen en entornos de producción. No son teoría: son las técnicas que usan los equipos de producto en empresas como Airbnb, Zillow o Spotahome para construir features con IA.

### Patrón 1 — Básico

El patrón más simple: una pregunta directa, sin contexto, sin instrucciones de rol. El modelo responde desde su conocimiento general.

**¿Cuándo usarlo?**  
- Exploración rápida o prototipado  
- Preguntas de conocimiento general donde no necesitas consistencia de formato  
- Chatbots conversacionales informales

**Limitación**: el modelo no sabe que está en Idealista, no conoce tu inventario, y puede dar respuestas genéricas que no se alinean con tu negocio.

In [2]:
# Patrón 1: Básico — una pregunta directa sin instrucciones adicionales
pregunta = "¿Cuáles son los factores más importantes al comprar un piso en España?"

respuesta_basica = cliente.messages.create(
    model=MODELO,
    max_tokens=500,
    messages=[
        {"role": "user", "content": pregunta}
    ]
)

print("=== PATRÓN BÁSICO ===")
print(f"Pregunta: {pregunta}\n")
print("Respuesta:")
print(respuesta_basica.content[0].text)

=== PATRÓN BÁSICO ===
Pregunta: ¿Cuáles son los factores más importantes al comprar un piso en España?

Respuesta:
# Factores clave al comprar un piso en España

## 📍 Ubicación
- **Zona y barrio**: servicios, transporte, seguridad
- Proximidad al trabajo, colegios, hospitales
- Planes urbanísticos futuros del municipio
- Nivel de ruido y contaminación

## 💰 Aspectos económicos
- Precio por m² comparado con la zona
- **Gastos adicionales**: ITP o IVA (10%), notaría, registro, gestoría (~10-15% extra)
- Hipoteca: tipo fijo vs variable, vinculaciones
- Comunidad de propietarios y derramas pendientes
- IBI y otros impuestos anuales

## 🏗️ Estado del inmueble
- Antigüedad y estado de conservación
- **Certificado energético** (afecta gastos futuros)
- Instalaciones: electricidad, fontanería, calefacción
- Posibles reformas necesarias

## 📋 Aspectos legales
- Cargas o hipotecas sobre el piso
- Nota simple del Registro de la Propiedad
- Situación catastral
- Estatutos de la comunidad
- **Cédul

### Patrón 2 — System Prompt

El **system prompt** es un mensaje especial que va *antes* de la conversación con el usuario. Define quién es el modelo, cómo debe comportarse, qué puede y qué no puede decir.

Piénsalo como el **briefing de onboarding** que le darías a un nuevo empleado:
- Quién eres
- Cuál es tu función
- Cómo debes comunicarte
- Cuáles son tus restricciones

**Para Idealista**: esto es lo que convierte un LLM genérico en "Carlos, nuestro asesor inmobiliario". El mismo modelo, resultados completamente diferentes.

**Cuándo usarlo**: casi siempre en producción. Es el patrón base para cualquier asistente con identidad propia.

In [3]:
# Patrón 2: System Prompt — definimos el rol y las reglas de comportamiento
system_carlos = """Eres Carlos, asesor inmobiliario senior con 15 años de experiencia 
en el mercado español. Trabajas para Idealista y ayudas a compradores a tomar 
decisiones informadas. Tus respuestas son concisas (máximo 200 palabras), 
prácticas y siempre terminan con una pregunta para entender mejor las necesidades 
del cliente."""

respuesta_carlos = cliente.messages.create(
    model=MODELO,
    max_tokens=500,
    system=system_carlos,
    messages=[
        {"role": "user", "content": pregunta}
    ]
)

print("=== PATRÓN SYSTEM PROMPT ===")
print(f"Sistema: {system_carlos[:80]}...\n")
print(f"Pregunta: {pregunta}\n")
print("Respuesta de Carlos:")
print(respuesta_carlos.content[0].text)

=== PATRÓN SYSTEM PROMPT ===
Sistema: Eres Carlos, asesor inmobiliario senior con 15 años de experiencia 
en el mercad...

Pregunta: ¿Cuáles son los factores más importantes al comprar un piso en España?

Respuesta de Carlos:
# Los factores clave al comprar un piso en España

Tras 15 años en el sector, te diría que hay **cinco pilares fundamentales**:

## 1. 📍 Ubicación
- Comunicaciones y transporte público
- Servicios cercanos (colegios, hospitales, comercios)
- Potencial de revalorización de la zona

## 2. 💰 Precio y financiación
- Precio por m² según mercado local
- Capacidad hipotecaria (máximo 30-35% ingresos)
- Gastos adicionales: **10-12% sobre el precio** (ITP/IVA, notaría, registro)

## 3. 🏗️ Estado del inmueble
- Antigüedad y reformas necesarias
- Certificado energético
- ITE (Inspección Técnica del Edificio)

## 4. ⚖️ Situación jurídica
- Cargas y deudas pendientes
- Estatutos de la comunidad
- Nota simple del Registro de la Propiedad

## 5. 🏘️ Comunidad de vecinos
- Cuotas 

### Patrón 3 — Few-Shot (Aprendizaje en Contexto)

**Few-shot** significa darle al modelo ejemplos de lo que queremos *dentro del mismo prompt*. El modelo aprende el patrón a partir de esos ejemplos y lo aplica al nuevo caso.

No es entrenamiento (no modificamos los pesos del modelo). Es **in-context learning**: el modelo lee los ejemplos en su memoria de trabajo y generaliza.

**Caso de uso en Idealista**: clasificar la intención del usuario cuando escribe en el buscador o en un chat. Esta clasificación puede luego enrutar al usuario al flujo correcto (compra vs alquiler, primera vivienda vs inversión).

**Por qué es poderoso**: no necesitas un clasificador entrenado. Con 3-5 ejemplos bien elegidos, el modelo puede clasificar con alta precisión.

In [4]:
# Patrón 3: Few-Shot — clasificación de intención con ejemplos en el prompt
# Categorías posibles para el negocio de Idealista
categorias = ["COMPRA_PRIMERA_VIVIENDA", "COMPRA_INVERSION", "ALQUILER_RESIDENCIAL", "ALQUILER_TEMPORAL", "VENTA"]

prompt_few_shot = f"""Clasifica la intención del usuario en una de estas categorías: {', '.join(categorias)}

Ejemplos:
Usuario: "Busco mi primer apartamento para vivir con mi pareja, tenemos ahorros para la entrada"
Intención: COMPRA_PRIMERA_VIVIENDA

Usuario: "Quiero comprar un local o piso pequeño para ponerlo en alquiler y obtener rentabilidad"
Intención: COMPRA_INVERSION

Usuario: "Necesito piso de alquiler en Madrid, presupuesto 900€/mes, contrato largo"
Intención: ALQUILER_RESIDENCIAL

Ahora clasifica este caso:
Usuario: "Estoy mirando pisos en Sevilla, no sé si comprar o alquilar, depende del precio"
Intención:"""

respuesta_fewshot = cliente.messages.create(
    model=MODELO,
    max_tokens=100,
    messages=[
        {"role": "user", "content": prompt_few_shot}
    ]
)

print("=== PATRÓN FEW-SHOT ===")
print("Categorías disponibles:", categorias)
print("\nNuevo caso: 'usuario indeciso en Sevilla'")
print("\nClasificación del modelo:")
print(respuesta_fewshot.content[0].text)

=== PATRÓN FEW-SHOT ===
Categorías disponibles: ['COMPRA_PRIMERA_VIVIENDA', 'COMPRA_INVERSION', 'ALQUILER_RESIDENCIAL', 'ALQUILER_TEMPORAL', 'VENTA']

Nuevo caso: 'usuario indeciso en Sevilla'

Clasificación del modelo:
## Clasificación

**Intención: INDETERMINADA / COMPRA_PRIMERA_VIVIENDA** *(ambigua)*

---

### Justificación

Este caso **no encaja claramente** en ninguna categoría definida porque:

- ❌ El usuario **no ha decidido** entre comprar o alquilar
- ❌ No menciona inversión ni rentabilidad
-


### Patrón 4 — Structured Output (Salida Estructurada)

Por defecto, los LLMs devuelven texto libre. Para integrar la respuesta en un sistema (base de datos, API, dashboard), necesitamos que salga en un **formato estructurado**: JSON, XML, etc.

Le pedimos al modelo que rellene un esquema concreto a partir de texto no estructurado (como una descripción de anuncio).

**Por qué JSON es mejor para sistemas**:
- Se puede parsear automáticamente
- Se puede validar contra un esquema
- Se puede insertar directamente en una base de datos
- Habilita pipelines de enriquecimiento de datos sin intervención humana

**Caso Idealista**: enriquecer fichas de propiedades automáticamente — extraer información estructurada de las descripciones libres que suben los agentes.

In [5]:
# Patrón 4: Structured Output — extraer datos estructurados de texto libre
descripcion_anuncio = """Precioso ático de 95m² en el barrio de Salamanca, Madrid. 
3 habitaciones amplias, 2 baños, cocina reformada con isla. Terraza privada de 20m² 
con vistas a la Sierra. Certificado energético B. Garaje incluido. 
Precio: 890.000€. Ideal para familia o inversión premium. 
Rentabilidad estimada por alquiler: 3.2% anual."""

esquema_json = """{
  "ubicacion": {"barrio": "", "ciudad": "", "vistas": ""},
  "caracteristicas": {"metros": 0, "habitaciones": 0, "banos": 0, "extras": []},
  "precio": {"valor": 0, "moneda": "EUR", "precio_m2": 0},
  "eficiencia": {"certificado_energetico": ""},
  "valoracion_inversion": {"rentabilidad_estimada_pct": 0, "perfil_comprador": ""}
}"""

prompt_structured = f"""Extrae la información del siguiente anuncio inmobiliario y devuelve ÚNICAMENTE un JSON válido con esta estructura exacta (sin texto adicional):

{esquema_json}

Anuncio:
{descripcion_anuncio}"""

respuesta_json = cliente.messages.create(
    model=MODELO,
    max_tokens=600,
    messages=[
        {"role": "user", "content": prompt_structured}
    ]
)

texto_json = respuesta_json.content[0].text
print("=== PATRÓN STRUCTURED OUTPUT ===")
print("Texto del anuncio (entrada):\n", descripcion_anuncio)
print("\nJSON extraído (salida):")
print(texto_json)

# Verificamos que el JSON es válido y parseable
try:
    datos = json.loads(texto_json)
    print("\nJSON parseado correctamente. Precio por m²:", datos.get('precio', {}).get('precio_m2', 'N/A'))
except json.JSONDecodeError:
    print("\nNota: ajusta el prompt si el modelo añade texto antes/después del JSON.")

=== PATRÓN STRUCTURED OUTPUT ===
Texto del anuncio (entrada):
 Precioso ático de 95m² en el barrio de Salamanca, Madrid. 
3 habitaciones amplias, 2 baños, cocina reformada con isla. Terraza privada de 20m² 
con vistas a la Sierra. Certificado energético B. Garaje incluido. 
Precio: 890.000€. Ideal para familia o inversión premium. 
Rentabilidad estimada por alquiler: 3.2% anual.

JSON extraído (salida):
```json
{
  "ubicacion": {"barrio": "Salamanca", "ciudad": "Madrid", "vistas": "Sierra"},
  "caracteristicas": {"metros": 95, "habitaciones": 3, "banos": 2, "extras": ["terraza privada 20m²", "cocina reformada con isla", "garaje"]},
  "precio": {"valor": 890000, "moneda": "EUR", "precio_m2": 9368},
  "eficiencia": {"certificado_energetico": "B"},
  "valoracion_inversion": {"rentabilidad_estimada_pct": 3.2, "perfil_comprador": "familia o inversor premium"}
}
```

Nota: ajusta el prompt si el modelo añade texto antes/después del JSON.


### Patrón 5 — Chain of Thought (Cadena de Razonamiento)

Le pedimos al modelo que **piense en voz alta**, paso a paso, antes de dar su conclusión. Esto produce resultados más precisos en tareas que requieren cálculo, razonamiento multi-paso o decisiones complejas.

**Por qué funciona**: los modelos de lenguaje son mejores cuando "escriben su razonamiento" antes de concluir. Es análogo a cómo un analista financiero llega a mejores resultados si muestra su trabajo en lugar de dar un número directo.

**Beneficio adicional para el negocio**: **auditabilidad**. Si el modelo recomienda una inversión, puedes ver exactamente cómo llegó a esa conclusión, cuáles fueron los números que usó, y si hay algún paso que no tiene sentido. Esto es crítico en contextos regulados o de alta responsabilidad.

**Caso Idealista**: evaluar si una propiedad es una buena inversión de forma transparente y verificable.

In [6]:
# Patrón 5: Chain of Thought — razonamiento explícito paso a paso
propiedad_inversion = """Piso en Valencia, precio de compra: 220.000€. 
Alquiler estimado: 950€/mes. Gastos anuales (IBI, comunidad, seguros, mantenimiento): 3.200€. 
Financiación: hipoteca al 3.5% a 25 años con 20% de entrada."""

prompt_cot = f"""Analiza si esta propiedad es una buena inversión inmobiliaria siguiendo EXACTAMENTE estos 7 pasos:

Paso 1: Calcula los ingresos brutos anuales por alquiler
Paso 2: Calcula los gastos anuales totales (incluye estimación de cuota hipotecaria)
Paso 3: Calcula los ingresos netos anuales
Paso 4: Calcula la rentabilidad neta sobre el precio de compra total
Paso 5: Estima el período de recuperación de la inversión
Paso 6: Compara con la rentabilidad de un depósito bancario al 3% anual
Paso 7: Da una conclusión final con recomendación clara (COMPRAR / NO COMPRAR / NEGOCIAR PRECIO)

Propiedad a analizar:
{propiedad_inversion}"""

respuesta_cot = cliente.messages.create(
    model=MODELO,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": prompt_cot}
    ]
)

print("=== PATRÓN CHAIN OF THOUGHT ===")
print("Propiedad a evaluar:")
print(propiedad_inversion)
print("\nAnálisis paso a paso:")
print(respuesta_cot.content[0].text)

=== PATRÓN CHAIN OF THOUGHT ===
Propiedad a evaluar:
Piso en Valencia, precio de compra: 220.000€. 
Alquiler estimado: 950€/mes. Gastos anuales (IBI, comunidad, seguros, mantenimiento): 3.200€. 
Financiación: hipoteca al 3.5% a 25 años con 20% de entrada.

Análisis paso a paso:
# Análisis de Inversión Inmobiliaria
## Piso en Valencia - 220.000€

---

## PASO 1: Ingresos Brutos Anuales por Alquiler

| Concepto | Cálculo | Resultado |
|----------|---------|-----------|
| Alquiler mensual | 950€ × 12 meses | **11.400€/año** |
| Ajuste vacancia (5%) | -570€ | |
| **Ingresos brutos reales** | | **10.830€/año** |

> ⚠️ Aplico un 5% de vacancia (±18 días/año) como estimación conservadora y realista

---

## PASO 2: Gastos Anuales Totales

### 2a. Cálculo de la hipoteca
- **Precio**: 220.000€
- **Entrada (20%)**: 44.000€
- **Capital financiado**: 176.000€
- **Tipo**: 3,5% a 25 años

**Cuota mensual ≈ 880€/mes → 10.560€/año**

> Fórmula aplicada: PMT con r=0,2917%/mes, n=300 cuotas

### 2b. Res

### Resumen comparativo de patrones

| Patrón | Complejidad | Cuándo usarlo | Caso Idealista |
|--------|-------------|---------------|----------------|
| **Básico** | Baja | Prototipado rápido, preguntas generales | FAQ genérico del sector |
| **System Prompt** | Baja-Media | Siempre en producción para dar identidad | Asistente "Carlos" o cualquier chatbot con marca |
| **Few-Shot** | Media | Clasificación, etiquetado, tareas con formato fijo | Clasificar intención de búsqueda, categorizar zonas |
| **Structured Output** | Media | Integración con sistemas, enriquecimiento de datos | Extracción automática de datos de anuncios |
| **Chain of Thought** | Alta | Decisiones complejas, cálculos, tareas auditables | Análisis de inversión, scoring de leads, validación de precio |

> **Regla práctica**: estos patrones se pueden combinar. El asistente de inversión ideal usaría *system prompt* (para dar identidad), *structured output* (para devolver datos parseables) y *chain of thought* (para justificar la recomendación).

## Parte 2: RAG — Retrieval Augmented Generation

### ¿Qué es RAG y por qué lo necesitamos?

Claude (o cualquier LLM) fue entrenado con datos hasta una fecha concreta. No conoce:
- Los pisos que tienes en tu inventario hoy
- Los precios de mercado de la semana pasada
- Los informes internos de tu empresa
- Los contratos de tus clientes

**RAG es la solución**: en lugar de reentrenar el modelo (caro, lento), le pasamos la información relevante *en el momento de la consulta*. El flujo es:

```
Pregunta del usuario
       ↓
Buscar documentos relevantes en nuestra base de datos
       ↓
Construir un prompt: "Dada esta información: [documentos], responde: [pregunta]"
       ↓
El modelo responde usando nuestros datos actuales
```

**Para Idealista**: RAG permite construir un asistente que responde sobre el inventario real, precios actuales y condiciones del mercado — sin exponer datos al modelo de forma permanente.

In [7]:
# Base de conocimiento: documentos inmobiliarios sintéticos
# En producción, estos vendrían de tu base de datos, CRM o sistema de anuncios

DOCUMENTOS_INMOBILIARIOS = [
    {
        "id": "DOC001",
        "titulo": "Piso en Malasaña, Madrid",
        "contenido": "Apartamento de 65m² en Malasaña, Madrid centro. 2 habitaciones, 1 baño, "
                     "cocina americana, suelos de madera. Precio: 380.000€ (5.846€/m²). "
                     "Certificado energético D. Comunidad: 120€/mes. Sin garaje. "
                     "Ideal para jóvenes profesionales o inversión en alquiler turístico."
    },
    {
        "id": "DOC002",
        "titulo": "Chalet en Pozuelo de Alarcón",
        "contenido": "Chalet independiente de 320m² en urbanización privada en Pozuelo de Alarcón. "
                     "5 habitaciones, 4 baños, piscina, jardín de 600m², garaje doble. "
                     "Precio: 1.250.000€. Certificado energético B. Zona escolar premium. "
                     "Perfil: familias con hijos, presupuesto alto."
    },
    {
        "id": "DOC003",
        "titulo": "Estudio en Barcelona, Eixample",
        "contenido": "Estudio reformado de 38m² en Eixample, Barcelona. 1 habitación tipo loft, "
                     "baño completo, balcón. Precio alquiler: 1.100€/mes. "
                     "Precio compra: 295.000€. Rentabilidad bruta alquiler: 4.5% anual. "
                     "Muy demandado por jóvenes y estudiantes de posgrado."
    },
    {
        "id": "DOC004",
        "titulo": "Informe de mercado: Madrid Q1 2025",
        "contenido": "El precio medio de venta en Madrid capital alcanzó 4.200€/m² en Q1 2025, "
                     "un incremento del 8.3% interanual. Los distritos con mayor subida: "
                     "Chamberí (+12%), Malasaña (+10.5%), Lavapiés (+9.8%). "
                     "El plazo medio de venta bajó a 45 días. Demanda extranjera: 18% del total."
    },
    {
        "id": "DOC005",
        "titulo": "Guía de rentabilidad: inversión inmobiliaria España",
        "contenido": "La rentabilidad bruta media del alquiler en España es del 5.1% (2024). "
                     "Ciudades más rentables: Murcia (7.2%), Valencia (6.8%), Sevilla (6.5%). "
                     "Madrid ofrece 4.2% y Barcelona 4.8%. "
                     "Para calcular rentabilidad neta, descontar gastos (IBI, comunidad, seguros, vacíos): aprox. 30-35% de ingresos brutos."
    },
    {
        "id": "DOC006",
        "titulo": "Piso de inversión en Valencia, Ruzafa",
        "contenido": "Piso de 75m² en barrio de Ruzafa, Valencia. 3 habitaciones, 2 baños, terraza 15m². "
                     "Precio compra: 260.000€. Alquiler actual: 1.400€/mes (inquilinos estables 3 años). "
                     "Rentabilidad bruta: 6.46%. Certificado energético C. "
                     "Zona en plena revalorización, alta demanda de jóvenes profesionales."
    }
]

print(f"Base de conocimiento cargada: {len(DOCUMENTOS_INMOBILIARIOS)} documentos")
for doc in DOCUMENTOS_INMOBILIARIOS:
    print(f"  [{doc['id']}] {doc['titulo']}")

Base de conocimiento cargada: 6 documentos
  [DOC001] Piso en Malasaña, Madrid
  [DOC002] Chalet en Pozuelo de Alarcón
  [DOC003] Estudio en Barcelona, Eixample
  [DOC004] Informe de mercado: Madrid Q1 2025
  [DOC005] Guía de rentabilidad: inversión inmobiliaria España
  [DOC006] Piso de inversión en Valencia, Ruzafa


### Vectorización TF-IDF

Para encontrar los documentos más relevantes para una pregunta, necesitamos **comparar textos matemáticamente**. TF-IDF es una técnica clásica (y efectiva) para esto.

**TF (Term Frequency)**: ¿Con qué frecuencia aparece una palabra en un documento? Si "rentabilidad" aparece 5 veces en un texto de 100 palabras, TF = 0.05.

**IDF (Inverse Document Frequency)**: ¿Cuántos documentos contienen esa palabra? Las palabras que aparecen en todos los documentos ("de", "en", "la") son poco informativas. Las palabras raras son más discriminativas.

**TF-IDF = TF × IDF**: pondera cada término por su importancia relativa.

**Similitud coseno**: para comparar dos textos, calculamos el ángulo entre sus vectores TF-IDF. Un ángulo de 0° significa textos idénticos (similitud = 1), un ángulo de 90° significa sin relación (similitud = 0).

In [8]:
# Implementación de TF-IDF y similitud coseno desde cero
# En producción usarías sklearn.TfidfVectorizer, pero esto muestra cómo funciona internamente

def tokenizar(texto):
    """Convierte texto a lista de tokens normalizados (minúsculas, sin puntuación)"""
    texto = texto.lower()
    tokens = re.findall(r'\b[a-záéíóúüñ]{3,}\b', texto)
    # Stopwords básicas en español
    stopwords = {'los', 'las', 'del', 'con', 'por', 'para', 'una', 'uno', 'que', 
                 'son', 'sus', 'más', 'muy', 'sin', 'hay', 'como', 'este', 'esta',
                 'ese', 'esa', 'una', 'dos', 'tres', 'año', 'años'}
    return [t for t in tokens if t not in stopwords]


def calcular_tfidf(documentos):
    """Calcula matriz TF-IDF para una lista de textos"""
    n_docs = len(documentos)
    
    # Tokenizar todos los documentos
    tokens_docs = [tokenizar(doc) for doc in documentos]
    
    # Vocabulario completo
    vocabulario = sorted(set(token for tokens in tokens_docs for token in tokens))
    
    # IDF: log(N / df(t)) para cada término
    idf = {}
    for termino in vocabulario:
        df = sum(1 for tokens in tokens_docs if termino in tokens)
        idf[termino] = math.log(n_docs / (1 + df))  # +1 para evitar división por cero
    
    # Matriz TF-IDF
    matriz = []
    for tokens in tokens_docs:
        n_tokens = len(tokens)
        conteo = Counter(tokens)
        vector = {}
        for termino in vocabulario:
            tf = conteo.get(termino, 0) / max(n_tokens, 1)
            vector[termino] = tf * idf[termino]
        matriz.append(vector)
    
    return matriz, vocabulario, idf


def similitud_coseno(vec_a, vec_b):
    """Calcula la similitud coseno entre dos vectores TF-IDF (dicts)"""
    terminos_comunes = set(vec_a.keys()) & set(vec_b.keys())
    
    producto_escalar = sum(vec_a[t] * vec_b[t] for t in terminos_comunes)
    norma_a = math.sqrt(sum(v**2 for v in vec_a.values()))
    norma_b = math.sqrt(sum(v**2 for v in vec_b.values()))
    
    if norma_a == 0 or norma_b == 0:
        return 0.0
    return producto_escalar / (norma_a * norma_b)


print("Funciones de vectorización definidas.")
print("Ejemplo de tokenización:")
ejemplo = "Piso de inversión con alta rentabilidad en Barcelona"
print(f"  Texto: '{ejemplo}'")
print(f"  Tokens: {tokenizar(ejemplo)}")

Funciones de vectorización definidas.
Ejemplo de tokenización:
  Texto: 'Piso de inversión con alta rentabilidad en Barcelona'
  Tokens: ['piso', 'inversión', 'alta', 'rentabilidad', 'barcelona']


### El AlmacenDocumentos

El `AlmacenDocumentos` es un **vector store en memoria**: una estructura que guarda los documentos junto con sus representaciones vectoriales para poder buscar por similitud semántica.

En producción, este almacén sería una base de datos vectorial como **Pinecone**, **Weaviate** o **pgvector** (extensión de PostgreSQL). Pero la lógica es exactamente la misma:

1. Guardar documentos con sus vectores
2. Cuando llega una consulta, vectorizarla también
3. Comparar el vector de la consulta con todos los vectores almacenados
4. Devolver los K documentos más similares

In [9]:
class AlmacenDocumentos:
    """
    Vector store en memoria para búsqueda semántica por TF-IDF.
    Equivalente simplificado de Pinecone, Weaviate o pgvector.
    """
    
    def __init__(self):
        self.documentos = []      # Lista de dicts con 'id', 'titulo', 'contenido'
        self.vectores = []         # Lista de vectores TF-IDF correspondientes
        self.vocabulario = []      # Vocabulario del corpus
        self.idf = {}              # Pesos IDF por término
    
    def agregar_documentos(self, documentos):
        """Indexa una lista de documentos y calcula sus vectores TF-IDF"""
        self.documentos = documentos
        textos = [f"{doc['titulo']} {doc['contenido']}" for doc in documentos]
        self.vectores, self.vocabulario, self.idf = calcular_tfidf(textos)
        print(f"Indexados {len(documentos)} documentos. Vocabulario: {len(self.vocabulario)} términos únicos.")
    
    def buscar(self, consulta, top_k=3):
        """Devuelve los top_k documentos más relevantes para la consulta"""
        # Vectorizar la consulta usando el mismo vocabulario e IDF del corpus
        tokens_consulta = tokenizar(consulta)
        conteo = Counter(tokens_consulta)
        n_tokens = len(tokens_consulta)
        
        vector_consulta = {}
        for termino in self.vocabulario:
            tf = conteo.get(termino, 0) / max(n_tokens, 1)
            vector_consulta[termino] = tf * self.idf.get(termino, 0)
        
        # Calcular similitud con cada documento
        puntuaciones = [
            (i, similitud_coseno(vector_consulta, vec))
            for i, vec in enumerate(self.vectores)
        ]
        
        # Ordenar por puntuación descendente y devolver top_k
        puntuaciones.sort(key=lambda x: x[1], reverse=True)
        
        resultados = []
        for idx, score in puntuaciones[:top_k]:
            doc = self.documentos[idx].copy()
            doc['score'] = round(score, 4)
            resultados.append(doc)
        
        return resultados


# Crear el almacén e indexar nuestros documentos
almacen = AlmacenDocumentos()
almacen.agregar_documentos(DOCUMENTOS_INMOBILIARIOS)

Indexados 6 documentos. Vocabulario: 105 términos únicos.


### Prueba de búsqueda vectorial

Antes de conectar con Claude, validamos que el sistema de recuperación funciona correctamente. Una buena búsqueda vectorial es la base de un buen RAG — si recuperamos documentos irrelevantes, el modelo no podrá responder bien aunque sea muy inteligente.

In [10]:
# Demo de búsqueda: comprobamos que el almacén recupera documentos relevantes
consultas_prueba = [
    "¿Cuál es la rentabilidad del alquiler en Valencia?",
    "Busco piso en Madrid con buena ubicación",
    "Precios del mercado inmobiliario en España"
]

for consulta in consultas_prueba:
    print(f"\nConsulta: '{consulta}'")
    resultados = almacen.buscar(consulta, top_k=2)
    for r in resultados:
        print(f"  [{r['score']:.3f}] {r['id']} — {r['titulo']}")


Consulta: '¿Cuál es la rentabilidad del alquiler en Valencia?'
  [0.299] DOC006 — Piso de inversión en Valencia, Ruzafa
  [0.223] DOC005 — Guía de rentabilidad: inversión inmobiliaria España

Consulta: 'Busco piso en Madrid con buena ubicación'
  [0.255] DOC001 — Piso en Malasaña, Madrid
  [0.255] DOC006 — Piso de inversión en Valencia, Ruzafa

Consulta: 'Precios del mercado inmobiliario en España'
  [0.284] DOC005 — Guía de rentabilidad: inversión inmobiliaria España
  [0.141] DOC004 — Informe de mercado: Madrid Q1 2025


### RAG en acción: recuperación + generación

La función `rag_query` orquesta el flujo completo:

1. **Recuperar**: busca en el almacén los documentos más relevantes para la pregunta
2. **Construir contexto**: formatea los documentos recuperados como texto
3. **Generar**: le pide a Claude que responda la pregunta *usando ese contexto*

La instrucción clave en el prompt es: *"Responde ÚNICAMENTE basándote en los documentos proporcionados"*. Esto evita que el modelo "alucine" información que no está en nuestros datos.

In [11]:
def rag_query(pregunta, almacen, top_k=3, verbose=False):
    """
    Ejecuta el flujo RAG completo:
    1. Recupera documentos relevantes del almacén
    2. Construye un prompt enriquecido con ese contexto
    3. Genera la respuesta con Claude
    
    Args:
        pregunta: pregunta del usuario
        almacen: instancia de AlmacenDocumentos
        top_k: número de documentos a recuperar
        verbose: si True, muestra los documentos recuperados
    """
    # Paso 1: Recuperar documentos relevantes
    docs_relevantes = almacen.buscar(pregunta, top_k=top_k)
    
    if verbose:
        print(f"  Documentos recuperados ({len(docs_relevantes)}):")
        for d in docs_relevantes:
            print(f"    [{d['score']:.3f}] {d['titulo']}")
    
    # Paso 2: Construir contexto con los documentos recuperados
    contexto = "\n\n".join([
        f"[{doc['id']}] {doc['titulo']}:\n{doc['contenido']}"
        for doc in docs_relevantes
    ])
    
    # Paso 3: Construir prompt RAG
    prompt_rag = f"""Eres un asesor inmobiliario experto de Idealista. 
Responde la pregunta del usuario ÚNICAMENTE basándote en los documentos proporcionados. 
Si la información no está en los documentos, dilo claramente.
Sé conciso y preciso. Cita el ID del documento cuando uses datos específicos.

DOCUMENTOS DE CONTEXTO:
{contexto}

PREGUNTA: {pregunta}"""
    
    # Paso 4: Generar respuesta con Claude
    respuesta = cliente.messages.create(
        model=MODELO,
        max_tokens=500,
        messages=[
            {"role": "user", "content": prompt_rag}
        ]
    )
    
    return respuesta.content[0].text


print("Función rag_query definida.")

Función rag_query definida.


### Comparativa: Con RAG vs Sin RAG

Aquí está la demostración clave: hacemos las mismas preguntas con y sin RAG, y comparamos las respuestas.

**Sin RAG**: el modelo responde desde su conocimiento general de entrenamiento — puede ser genérico, desactualizado, o inventar datos específicos.

**Con RAG**: el modelo responde basándose en *nuestros documentos* — las respuestas son específicas, verificables, y alineadas con nuestro inventario real.

In [12]:
# Preguntas para comparar RAG vs sin RAG
preguntas_comparativa = [
    "¿Cuánto cuesta un piso en Malasaña, Madrid?",
    "¿Qué rentabilidad puedo esperar del alquiler en Valencia?",
    "¿Cuál es la evolución reciente del mercado inmobiliario en Madrid?"
]

for i, pregunta in enumerate(preguntas_comparativa, 1):
    print(f"\n{'='*60}")
    print(f"PREGUNTA {i}: {pregunta}")
    print('='*60)
    
    # --- SIN RAG ---
    respuesta_sin_rag = cliente.messages.create(
        model=MODELO,
        max_tokens=200,
        messages=[
            {"role": "user", "content": pregunta}
        ]
    )
    
    print("\n[SIN RAG — conocimiento general del modelo]:")
    print(respuesta_sin_rag.content[0].text)
    
    # --- CON RAG ---
    print("\n[CON RAG — usando nuestros documentos]:")
    respuesta_con_rag = rag_query(pregunta, almacen, top_k=2, verbose=True)
    print(respuesta_con_rag)


PREGUNTA 1: ¿Cuánto cuesta un piso en Malasaña, Madrid?

[SIN RAG — conocimiento general del modelo]:
## Precios de pisos en Malasaña, Madrid

Los precios pueden variar bastante según el tipo de operación:

### 🏠 Compra
- **Precio medio aproximado**: entre **4.000 y 6.500 €/m²**
- Un piso de **60-70 m²** puede costar entre **280.000 y 450.000 €**
- Los más céntricos o reformados pueden superar fácilmente los **500.000 €**

### 🔑 Alquiler
- **Estudio o 1 habitación**: desde **900 € hasta 1.400 €/mes**
- **2 habitaciones**: entre **1.400 y 2.200 €/mes**
- **3 habitaciones**: desde **2.000 

[CON RAG — usando nuestros documentos]:
  Documentos recuperados (2):
    [0.422] Piso en Malasaña, Madrid
    [0.193] Piso de inversión en Valencia, Ruzafa
## Piso en Malasaña, Madrid

Según la información disponible **[DOC001]**, el piso en Malasaña tiene un precio de:

- **Precio total: 380.000 €**
- **Precio por m²: 5.846 €/m²**

### Características principales:
- 65 m² | 2 habitaciones | 1 baño


### ¿Por qué RAG? Ventajas clave

- **Actualización en tiempo real**: los documentos en el almacén se pueden actualizar sin tocar el modelo. Nuevos anuncios, precios revisados, informes del mes — el asistente los usa automáticamente.

- **Verificabilidad**: cada respuesta está anclada a documentos específicos (con ID). Puedes auditar qué fuente usó el modelo y detectar errores en los datos de origen.

- **Reducción de alucinaciones**: al obligar al modelo a responder *solo con el contexto proporcionado*, eliminamos las respuestas inventadas. Si la información no está en el almacén, el modelo lo dice.

- **Privacidad y control**: los datos nunca salen de tu infraestructura de forma permanente. El modelo no "aprende" tu inventario — solo lo usa en cada consulta. Esto es crítico para datos sensibles (precios, contratos, clientes).

- **Escala sin reentrenamiento**: puedes tener millones de documentos en el almacén vectorial. Añadir nuevas propiedades o informes es tan simple como insertarlos en la base de datos, sin ningún proceso de ML.